In [ ]:
#setup and cofigs
import json
import re
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, explode, udf, to_json, struct
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType

with open("config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

BOOTSTRAP_SERVER = config["kafka"]["bootstrap_server"]
RAW_TOPIC = config["kafka"]["raw_topic"]
TOKENS_TOPIC = config["kafka"]["tokens_topic"]
FOLLOWED_TOKENS = config["followed_tokens"]

spark = SparkSession.builder \
    .appName("TelegramTokenizer") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark session initialized.")

In [ ]:
# Raw JSON schema as produced by the Telethon producer
raw_schema = StructType([
    StructField("channel", StringType(), True),
    StructField("text", StringType(), True),
    StructField("ts", LongType(), True)
])

# a simple worker functaion for parsing one message into tokes
def tokenize(text):
    if not text:
        return []
    #Hebrew and alphanumeric words of length >= 2
    return re.findall(r"[\u0590-\u05fe\w]{2,}", text.lower())

#turn to a udf
tokenize_udf = udf(tokenize_message, ArrayType(StringType()))

In [ ]:
# connect to kafka topic
raw_stream_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("subscribe", RAW_TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

# parse data into json and then into data frame
parsed_df = raw_stream_df.select(
    from_json(col("value").cast("string"), raw_schema).alias("data")
).select("data.*")

#tokenize, explode, and filter, in parallel
tokens_df = parsed_df \
    .withColumn("word", explode(tokenize_udf(col("text")))) \
    .filter(col("word").isin(FOLLOWED_TOKENS))

#parse into the kafka token stream format:: key: token, value: timestamp
kafka_output_df = tokens_df.select(
    col("word").alias("key"),
    to_json(struct(col("ts"))).alias("value")
)

#stream to kafka secend producer
query = kafka_output_df.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("topic", TOKENS_TOPIC) \
    .option("checkpointLocation", "/tmp/spark-kafka-tokenizer-checkpoint") \
    .outputMode("append") \
    .start()

print(f"Stage 1 Spark Streaming is live!")
print(f"Reading from '{RAW_TOPIC}' ➔ Tokenizing in parallel ➔ Keyed output to '{TOKENS_TOPIC}'")

query.awaitTermination()

In [1]:
#"level"
import math
def update_level(current_level, arrivals, baseline):
    thresh_hold = max(7,3*math.sqrt(baseline))
    r = 2 #can be alterd, the level sinks twice as fast as the baseline

    current_level = max(0, current_level + arrivals - r * baseline)
    is_alert = (current_level >= thresh_hold)

    return current_level, is_alert


In [10]:
level = 0
test_stream = [500000, 600000, 400000, 800000, 1000007, 500000, 200000]

for arrivals in test_stream:
    level, is_alert = update_level(level, arrivals, baseline=500000)
    print(f"Arrivals: {arrivals:2d} | New Level: {level:4.1f} | Alert: {is_alert}")

Arrivals: 500000 | New Level:  0.0 | Alert: False
Arrivals: 600000 | New Level:  0.0 | Alert: False
Arrivals: 400000 | New Level:  0.0 | Alert: False
Arrivals: 800000 | New Level:  0.0 | Alert: False
Arrivals: 1000007 | New Level:  7.0 | Alert: False
Arrivals: 500000 | New Level:  0.0 | Alert: False
Arrivals: 200000 | New Level:  0.0 | Alert: False
